# PathLDM - 이미지 생성 (BR 5000장 + ST 5000장)

학습된 PathLDM 체크포인트에서 이미지를 생성합니다.
- BR 클래스 (BRDC, BRID, BRIL, BRLC, BRNT): 각 1000장 = **5000장**
- ST 클래스 (STDI, STIN, STMX, STNT): 각 1250장 = **5000장**
- 저장 경로: `../../../results/NIPA/pathldm/{클래스명}/`
- 체크포인트: `../../../model/NIPA/pathldm/model-35.pt` (최신)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 1. Import 및 설정

In [ ]:
import sys
import math
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from tqdm import tqdm
from diffusers import AutoencoderKL

# LDM 모듈 경로
LDM_DIR = os.path.abspath(".")
if LDM_DIR not in sys.path:
    sys.path.insert(0, LDM_DIR)

from ldm.modules.diffusionmodules.openaimodel import UNetModel
from ldm.modules.diffusionmodules.util import make_beta_schedule
from ldm.modules.ema import LitEma

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── 경로 ──
CHECKPOINT_PATH = "../../../model/NIPA/pathldm/model-35.pt"
SAVE_ROOT = "../../../results/NIPA/pathldm"
os.makedirs(SAVE_ROOT, exist_ok=True)

# ── 모델 아키텍처 (학습 시와 동일) ──
IMAGE_SIZE = 1024
LATENT_SIZE = 128
LATENT_CHANNELS = 4
SCALE_FACTOR = 0.18215

MODEL_CHANNELS = 256
CHANNEL_MULT = [1, 2, 4]
ATTENTION_RESOLUTIONS = [4, 2]
NUM_RES_BLOCKS = 2
NUM_HEAD_CHANNELS = 32
CONTEXT_DIM = 512
TRANSFORMER_DEPTH = 1

# ── Diffusion schedule ──
TIMESTEPS = 1000
LINEAR_START = 0.0015
LINEAR_END = 0.0195

# ── 클래스 정의 ──
CLASS_TO_IDX = {
    "BRDC": 1, "BRID": 2, "BRIL": 3, "BRLC": 4, "BRNT": 5,
    "STDI": 6, "STIN": 7, "STMX": 8, "STNT": 9,
}
N_EMBED = 10  # 9 classes + 1 unconditional

# ── 생성 설정 ──
GUIDANCE_SCALE = 3.0
DDIM_STEPS = 50
BATCH_SIZE = 4  # 한 번에 생성할 이미지 수

# BR 5000장: 5클래스 × 1000장
# ST 5000장: 4클래스 × 1250장
GEN_PLAN = {
    "BRDC": 1000, "BRID": 1000, "BRIL": 1000, "BRLC": 1000, "BRNT": 1000,
    "STDI": 1250, "STIN": 1250, "STMX": 1250, "STNT": 1250,
}

total = sum(GEN_PLAN.values())
br_total = sum(v for k, v in GEN_PLAN.items() if k.startswith("BR"))
st_total = sum(v for k, v in GEN_PLAN.items() if k.startswith("ST"))
print(f"생성 계획: BR {br_total}장 + ST {st_total}장 = 총 {total}장")
print(f"저장 경로: {os.path.abspath(SAVE_ROOT)}")

## 2. 모델 로드

In [ ]:
# ── Noise schedule ──
betas = torch.tensor(
    make_beta_schedule("linear", TIMESTEPS, linear_start=LINEAR_START, linear_end=LINEAR_END),
    dtype=torch.float32,
)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0).to(device)

print(f"Noise schedule loaded: T={TIMESTEPS}")

In [ ]:
# ── ClassEmbedder ──
class ClassEmbedder(nn.Module):
    def __init__(self, embed_dim, n_classes):
        super().__init__()
        self.embedding = nn.Embedding(n_classes, embed_dim)

    def forward(self, labels):
        return self.embedding(labels.unsqueeze(1))


# ── UNet ──
unet = UNetModel(
    image_size=LATENT_SIZE,
    in_channels=LATENT_CHANNELS,
    out_channels=LATENT_CHANNELS,
    model_channels=MODEL_CHANNELS,
    attention_resolutions=ATTENTION_RESOLUTIONS,
    num_res_blocks=NUM_RES_BLOCKS,
    channel_mult=CHANNEL_MULT,
    num_head_channels=NUM_HEAD_CHANNELS,
    use_spatial_transformer=True,
    transformer_depth=TRANSFORMER_DEPTH,
    context_dim=CONTEXT_DIM,
    use_checkpoint=False,  # 추론 시 불필요
).to(device)

# ── ClassEmbedder ──
class_embedder = ClassEmbedder(embed_dim=CONTEXT_DIM, n_classes=N_EMBED).to(device)

# ── 체크포인트 로드 ──
print(f"Loading checkpoint: {CHECKPOINT_PATH}")
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
print(f"  Step: {ckpt['step']}")

# UNet 가중치 로드 후 EMA 적용
unet.load_state_dict(ckpt["unet"])
class_embedder.load_state_dict(ckpt["embedder"])

# EMA 가중치를 UNet에 적용
ema = LitEma(unet, decay=0.9999)
ema.load_state_dict(ckpt["ema"])
ema.copy_to(unet)  # copy_to는 모델 객체를 받음 (parameters() generator가 아님)
print("  EMA weights applied to UNet")

unet.eval()
class_embedder.eval()

# ── VAE (frozen) ──
print("Loading VAE...")
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()

del ckpt, ema
torch.cuda.empty_cache()
print(f"Models loaded. UNet params: {sum(p.numel() for p in unet.parameters()) / 1e6:.1f}M")

## 3. 샘플링 함수 정의

In [ ]:
def vae_decode(z):
    """latent -> image [0,1]"""
    with torch.no_grad():
        decoded = vae.decode(z / SCALE_FACTOR).sample
    return torch.clamp((decoded + 1.0) / 2.0, 0.0, 1.0)


@torch.no_grad()
def ddim_sample_cfg(labels, guidance_scale=3.0, ddim_steps=50):
    """Classifier-Free Guidance DDIM sampling"""
    B = labels.shape[0]

    uncond_labels = torch.zeros(B, dtype=torch.long, device=device)
    context_cond   = class_embedder(labels)
    context_uncond = class_embedder(uncond_labels)
    context = torch.cat([context_uncond, context_cond], dim=0)

    step_size = TIMESTEPS // ddim_steps
    timesteps = list(reversed(range(0, TIMESTEPS, step_size)))

    z = torch.randn(B, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE, device=device)

    for i, t_val in enumerate(timesteps):
        t_batch = torch.full((B,), t_val, device=device, dtype=torch.long)
        t_batch_2 = torch.cat([t_batch, t_batch], dim=0)
        z_2 = torch.cat([z, z], dim=0)

        eps_both = unet(z_2, t_batch_2, context=context)
        eps_uncond, eps_cond = eps_both.chunk(2)
        eps_guided = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        alpha_t = alphas_cumprod[t_val]
        t_prev_val = timesteps[i + 1] if i + 1 < len(timesteps) else -1
        alpha_t_prev = alphas_cumprod[t_prev_val] if t_prev_val >= 0 else torch.ones(1, device=device)

        z0_pred = (z - (1 - alpha_t).sqrt() * eps_guided) / alpha_t.sqrt()
        z0_pred = z0_pred.clamp(-1.0, 1.0)

        dir_xt = (1 - alpha_t_prev).sqrt() * eps_guided
        z = alpha_t_prev.sqrt() * z0_pred + dir_xt

    return vae_decode(z)


def save_images(images, class_name, start_idx):
    """텐서 이미지 배치를 개별 PNG로 저장"""
    save_dir = os.path.join(SAVE_ROOT, class_name)
    os.makedirs(save_dir, exist_ok=True)
    for i, img in enumerate(images):
        img_np = (img.cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        Image.fromarray(img_np).save(
            os.path.join(save_dir, f"{class_name}_{start_idx + i:05d}.png")
        )


print("Sampling functions defined.")

## 4. 이미지 생성 (BR 5000 + ST 5000 = 10000장)

In [ ]:
print(f"=" * 60)
print(f"이미지 생성 시작")
print(f"Guidance scale: {GUIDANCE_SCALE}, DDIM steps: {DDIM_STEPS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"=" * 60)

total_generated = 0

for class_name, n_images in GEN_PLAN.items():
    class_idx = CLASS_TO_IDX[class_name]
    n_batches = math.ceil(n_images / BATCH_SIZE)
    generated = 0

    print(f"\n[{class_name}] idx={class_idx}, 목표: {n_images}장")

    for batch_i in tqdm(range(n_batches), desc=class_name):
        current_batch = min(BATCH_SIZE, n_images - generated)
        labels = torch.full((current_batch,), class_idx, dtype=torch.long, device=device)

        images = ddim_sample_cfg(labels, guidance_scale=GUIDANCE_SCALE, ddim_steps=DDIM_STEPS)
        save_images(images, class_name, start_idx=generated)

        generated += current_batch
        total_generated += current_batch

    print(f"  -> {generated}장 저장 완료: {os.path.join(SAVE_ROOT, class_name)}")

print(f"\n{'=' * 60}")
print(f"전체 생성 완료: {total_generated}장")
print(f"저장 경로: {os.path.abspath(SAVE_ROOT)}")

## 5. 생성 결과 확인

In [ ]:
import matplotlib.pyplot as plt

# 각 클래스별 생성 수 확인
print("클래스별 생성 이미지 수:")
for class_name in GEN_PLAN:
    class_dir = os.path.join(SAVE_ROOT, class_name)
    if os.path.exists(class_dir):
        count = len([f for f in os.listdir(class_dir) if f.endswith(".png")])
        print(f"  {class_name}: {count}장")

# 각 클래스에서 1장씩 시각화
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, (class_name, _) in enumerate(GEN_PLAN.items()):
    row, col = i // 5, i % 5
    class_dir = os.path.join(SAVE_ROOT, class_name)
    sample_file = sorted(os.listdir(class_dir))[0] if os.path.exists(class_dir) else None
    if sample_file:
        img = Image.open(os.path.join(class_dir, sample_file))
        axes[row][col].imshow(img)
    axes[row][col].set_title(class_name)
    axes[row][col].axis("off")

# 마지막 빈 칸 숨기기
axes[1][4].axis("off")

plt.suptitle("PathLDM Generated Samples (1 per class)", fontsize=14)
plt.tight_layout()
plt.show()